# orbit-agent on a GPU

Runs `python -m src.model.retrain --embed siglip`.
Code: https://github.com/devxsubh/orbit-agent.git (branch `restructure`). Photos: the real Sentinel-2A replay (private dataset).
Results are saved in `results/` (Output tab).

In [ ]:
# 1. Get the project code (into /tmp, so it isn't copied into the output)
import glob, os, shutil, subprocess, time

def run(command):
    """Run a shell command and STOP the notebook if it fails."""
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)

run("rm -rf /tmp/orbit-agent && git clone --depth 1 -b restructure https://github.com/devxsubh/orbit-agent.git /tmp/orbit-agent")
os.chdir("/tmp/orbit-agent")
run("git log --oneline -1")
run("pip install -q skyfield ollama")  # not on Kaggle (it has torch, transformers, sklearn)
time.sleep(1)
started = time.time()  # results = files written after this (not the ones that came with the code)

In [ ]:
# 2. Which GPU did we get? Where are the photos?
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
found = glob.glob("/kaggle/input/**/AID/Airport", recursive=True)
if found:
    os.environ["AID_DIR"] = os.path.dirname(found[0])
    print("AID photos in", os.environ["AID_DIR"])
found = glob.glob("/kaggle/input/**/chips.csv", recursive=True)
if found:
    os.environ["S2_REAL_DIR"] = os.path.dirname(found[0])
    print("Real Sentinel-2A photos in", os.environ["S2_REAL_DIR"])

In [ ]:
# 3. The actual work
run("python -u -m src.model.retrain --embed siglip")

In [ ]:
# 4. Keep the results (Kaggle saves everything in /kaggle/working)
os.makedirs("/kaggle/working/results", exist_ok=True)
files = [f for f in glob.glob("results/*/*.csv") + glob.glob("data/processed/embeddings_*.npz")
         if os.path.getmtime(f) > started]  # only what THIS run made
assert files, "No results were made"
for f in files:
    shutil.copy(f, "/kaggle/working/results/")
run("ls -lh /kaggle/working/results")